# W07 -- Data Summary: Target Definition, Feature Engineering, and Modeling

This notebook takes the R-joined metro panel and:

1. Defines the **affordability-collapse** target (`collapse_onset`).
2. Engineers lagged, leakage-safe predictive features.
3. Trains and evaluates two XGBoost classifiers:
   - **Model 1 (explanatory):** Austin, Boise, Tampa -- 4 features -- used for SHAP feature-importance analysis.
   - **Model 2 (generalization/scoring):** Austin, Boise (Tampa excluded, see Feature Engineering section) -- 3 features -- tuned with Optuna, then used to score every other metro (the "holdout" set) for early-warning risk.
4. Backtests both models with a logistic-regression check.

**Input:** `data/final_data/price_changes_with_collapse_flags.csv` (produced by the R data-join pipeline in the `.rmd` sibling of this notebook).

**Outputs:** train/val/holdout CSV splits, metrics tables, and SHAP/risk-ranking figures under `output/`.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
import optuna

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, cross_val_predict
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report, make_scorer, precision_recall_curve
)
from statsmodels.discrete.discrete_model import Logit

In [ ]:
# Load the R-joined panel (CBSA is the join key across all 5 data sources)
df = pd.read_csv("../../data/final_data/price_changes_with_collapse_flags.csv")

# R exports columns like "metro_name.x" -- flatten dots to underscores for easier access
df.columns = df.columns.str.replace('.', '_', regex=False)

print(df.shape)
print(df.dtypes[['cbsa', 'metro_name_x', 'year', 'qtr']])

## Target definition: affordability collapse

`collapse_onset` marks the first quarter a metro's price-to-income ratio crosses above a threshold and stays there (an "unaffordable" regime). We use **5.0** as that threshold -- chosen below by comparing onset dates at 4.0 / 4.5 / 5.0 / 5.5 and picking the value that gives the tightest, most realistic cluster of onset dates across the three training cities.

In [ ]:
# Confirms Austin (12420), Boise (14260), and Tampa (45294) all have data,
# and that the onset dates line up with what we expect: Austin 2021Q2, Boise 2019Q3, Tampa 2021Q4.
training_cbsa_map = {'Austin': 12420.0, 'Boise': 14260.0, 'Tampa': 45294.0}

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

df['is_unaffordable'] = df['price_to_income_ratio'] > 5.0
df['prev_unaffordable'] = g['is_unaffordable'].shift(1).fillna(False).astype(bool)
df['collapse_onset'] = df['is_unaffordable'] & (~df['prev_unaffordable'])

first_collapse = (
    df[df['collapse_onset']]
    .groupby('cbsa')[['metro_name_x', 'year', 'qtr', 'price_to_income_ratio']]
    .first()
)
print("Collapse onset check (training cities):")
print(first_collapse.loc[first_collapse.index.isin(training_cbsa_map.values())])
# catches silent join failures before they cause problems in feature engineering

In [ ]:
# Sensitivity check: does the onset date move much if the threshold isn't 5.0?
# Goal: confirm 5.0 isn't an arbitrary choice -- see if a different threshold changes results significantly.
for threshold in [4.0, 4.5, 5.0, 5.5]:
    print(f"\n--- Threshold: {threshold} ---")
    for city, code_ in training_cbsa_map.items():
        sub = df[df['cbsa'] == code_].sort_values(['year', 'qtr']).copy()
        sub = sub[sub['price_to_income_ratio'].notna()]
        sub['is_unaffordable'] = sub['price_to_income_ratio'] > threshold
        sub['prev_unaffordable'] = sub['is_unaffordable'].shift(1).fillna(False)
        sub['onset'] = sub['is_unaffordable'] & ~sub['prev_unaffordable']
        onset_row = sub[sub['onset']].head(1)
        if len(onset_row) > 0:
            print(f"  {city}: {onset_row['year'].values[0]}Q{onset_row['qtr'].values[0]}")
        else:
            print(f"  {city}: never crosses this threshold")

# Result: 5.0 gives the tightest, most realistic cluster of onset dates for all three training cities.

## Feature engineering

Every feature below is lagged by 4 quarters (1 year) before any rolling calculation. This ensures no feature accidentally "sees" the same-quarter data used to build `collapse_onset`.

This section creates lagged and trend-based indicators from housing, income, and population data. Lagged features ensure the model only uses information available before the period being predicted, which prevents data leakage.

**Key indicators:**
* Price-to-income ratio (5-year change)
* Recent home-price growth (3-year rolling trend, QoQ change)
* Prior-period house-price appreciation (HPI YoY)
* Population velocity and acceleration

**Rule:** never let a feature use current- or future-period information that overlaps with label construction.

In [ ]:
LAG = 4

df = df.sort_values(['cbsa', 'year', 'qtr']).reset_index(drop=True)
g = df.groupby('cbsa')

# Feature 1: price-to-income 5-year change
df['price_to_income_lag'] = g['price_to_income_ratio'].shift(LAG)
df['price_to_income_5yr_chg'] = df.groupby('cbsa')['price_to_income_lag'].transform(
    lambda x: x - x.shift(20)
)

# Feature 2: affordability momentum -- 3yr rolling slope of lagged ZHVI YoY change
df['zhvi_yoy_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(4))
df['zhvi_qoq_fixed'] = df.groupby('cbsa')['zhvi_qtr'].transform(lambda x: x.pct_change(1))
df['zhvi_yoy_lag'] = df.groupby('cbsa')['zhvi_yoy_fixed'].shift(LAG)
df['three-year_home_price_growth_trend'] = df.groupby('cbsa')['zhvi_yoy_lag'].transform(
    lambda x: x.rolling(12, min_periods=12).apply(
        lambda y: np.polyfit(np.arange(len(y)), y, 1)[0] if y.notna().all() else np.nan
    )
)
df['zhvi_qoq_lag'] = df.groupby('cbsa')['zhvi_qoq_fixed'].shift(LAG)

# Feature 3: HPI-based momentum (YoY change in the seasonally-adjusted index)
df['hpi_yoy'] = df.groupby('cbsa')['index_sa'].transform(lambda x: x.pct_change(4))
df['hpi_yoy_lag'] = df.groupby('cbsa')['hpi_yoy'].shift(LAG)

# Secondary signal (not in the core feature set, kept for reference)
df['hpi_3yr_chg'] = g['index_sa'].transform(lambda x: (x / x.shift(12)) - 1) * 100

# Features 4 & 5: population velocity and acceleration, lagged
# Note: population coverage has a known gap for Tampa, so these features are used
# for Austin/Boise and the broader holdout set, but not for the Tampa-inclusive model.
df["pop_yoy_fixed"] = g["population"].transform(lambda x: x.pct_change(4))
df["pop_velocity_fixed"] = df.groupby("cbsa")["pop_yoy_fixed"].diff()
df["pop_velocity_lag"] = df.groupby("cbsa")["pop_velocity_fixed"].shift(LAG)
df["pop_acceleration_fixed"] = df.groupby("cbsa")["pop_velocity_fixed"].diff()
df["pop_acceleration_lag"] = df.groupby("cbsa")["pop_acceleration_fixed"].shift(LAG)

In [ ]:
# Coverage check: confirm every engineered column exists and see how many
# non-null rows/cities each feature has for the training cities vs. the holdout set.
train_features = ["three-year_home_price_growth_trend", "zhvi_qoq_lag", "price_to_income_5yr_chg", "hpi_yoy_lag"]
pop_features = ["pop_velocity_lag", "pop_acceleration_lag"]

missing_cols = [c for c in train_features + pop_features + ["collapse_onset"] if c not in df.columns]
print("Missing engineered columns:", missing_cols)

print("\nTraining city coverage -- 4 core features:")
for city, code_ in training_cbsa_map.items():
    sub = df[df["cbsa"] == code_]
    coverage = {col: int(sub[col].notna().sum()) for col in train_features}
    print(f"  {city}: {coverage}")

print("\nTraining city coverage -- population features:")
for city, code_ in training_cbsa_map.items():
    sub = df[df["cbsa"] == code_]
    print(f"  {city}: pop_velocity_lag={sub['pop_velocity_lag'].notna().sum()}, "
          f"pop_acceleration_lag={sub['pop_acceleration_lag'].notna().sum()}")

holdout_check = df[~df["cbsa"].isin(training_cbsa_map.values())]
print("\nHoldout coverage -- 4 core features:")
for col in train_features:
    nonnull = holdout_check[col].notna()
    print(f"  {col}: {nonnull.sum()} rows, {holdout_check[nonnull]['cbsa'].nunique()} cities")

print("\nHoldout coverage -- population features:")
for col in pop_features:
    nonnull = holdout_check[col].notna()
    print(f"  {col}: {nonnull.sum()} rows, {holdout_check[nonnull]['cbsa'].nunique()} cities")

print("\nHoldout price_to_income_ratio non-null:", holdout_check["price_to_income_ratio"].notna().sum())
print("Holdout collapse_onset True count:", holdout_check["collapse_onset"].sum())

## Findings: label coverage

`price_to_income_ratio` -- and therefore the `collapse_onset` label -- has ground-truth values only for Austin, Boise, and Tampa; no other metro has income data at this granularity. `hpi_yoy_lag` and the population features, however, have broad multi-city coverage, so they can be used to **score** risk in other metros, but those scores can't be validated against a true label. This is consistent with the early-warning framing of the research question: for untrained metros we're ranking relative risk, not claiming certainty.

In [ ]:
# Create output directory if it doesn't exist
os.makedirs("output", exist_ok=True)

# Model 1 data: explanatory (3 training cities, 4 features)
# filtering to rows for the 3 training cities with no missing feature/target values
model1_df = df[df["cbsa"].isin(training_cbsa_map.values())].dropna(
    subset=train_features + ["collapse_onset"]
).copy()

# running total of collapse_onset per city, used to split pre- vs post-collapse periods
model1_df["cum_onset"] = model1_df.groupby("cbsa")["collapse_onset"].cumsum()

# training set: only rows before the first collapse_onset flag (cum_onset still 0)
model1_train = model1_df[model1_df["cum_onset"] == 0]

# validation set: rows at or after the collapse_onset event (cum_onset > 0)
model1_val = model1_df[model1_df["cum_onset"] > 0]

print("Model 1 -- training rows (pre-collapse, 3 cities):", len(model1_train))
print("Model 1 -- validation rows (collapse onset+, 3 cities):", len(model1_val))

model1_train.to_csv("output/model1_train.csv", index=False)
model1_val.to_csv("output/model1_val.csv", index=False)

In [ ]:
# Model 2 data: generalization/scoring (Austin + Boise, 3 features)
# reduces the feature set used for the scoring model (drops one feature vs. Model 1)
scoring_features = ["hpi_yoy_lag", "pop_velocity_lag", "pop_acceleration_lag"]
# only Austin and Boise this time (Tampa excluded -- see population coverage gap above)
model2_train_cities = {"Austin": training_cbsa_map["Austin"], "Boise": training_cbsa_map["Boise"]}

model2_df = df[df["cbsa"].isin(model2_train_cities.values())].dropna(
    subset=scoring_features + ["collapse_onset"]
).copy()

model2_df["cum_onset"] = model2_df.groupby("cbsa")["collapse_onset"].cumsum()
model2_train = model2_df[model2_df["cum_onset"] == 0]
model2_val = model2_df[model2_df["cum_onset"] > 0]

print("\nModel 2 -- training rows (Austin+Boise pre-collapse):", len(model2_train))
print("Model 2 -- validation rows (Austin+Boise collapse onset+):", len(model2_val))

model2_train.to_csv("output/model2_train.csv", index=False)
model2_val.to_csv("output/model2_val.csv", index=False)

In [ ]:
# Holdout score set: all other metros, 3 features, no ground-truth labels.
# Excludes the 3 training cities -- everything else is a candidate for scoring.
holdout_scoring = df[~df["cbsa"].isin(training_cbsa_map.values())].dropna(
    subset=scoring_features
).copy()

print("\nHoldout scoring rows:", len(holdout_scoring), "| cities:", holdout_scoring["cbsa"].nunique())

holdout_scoring.to_csv("output/holdout_scoring.csv", index=False)
print("\nSaved: model1_train.csv, model1_val.csv, model2_train.csv, model2_val.csv, holdout_scoring.csv")

## Modeling: XGBoost + SHAP explainability

In [ ]:
os.makedirs("output/figures", exist_ok=True)
os.makedirs("output/tables", exist_ok=True)

train_features = ["three-year_home_price_growth_trend", "zhvi_qoq_lag", "price_to_income_5yr_chg", "hpi_yoy_lag"]
scoring_features = ["hpi_yoy_lag", "pop_velocity_lag", "pop_acceleration_lag"]

In [ ]:
# Reload from the saved splits so this modeling section can be re-run independently
# of the feature-engineering cells above.
model1_train = pd.read_csv("output/model1_train.csv")
model1_val = pd.read_csv("output/model1_val.csv")
model2_train = pd.read_csv("output/model2_train.csv")
model2_val = pd.read_csv("output/model2_val.csv")
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")

### Model 1: Early-Warning Indicator Model

Model 1 identifies which engineered housing-market indicators are most associated with the historical affordability-risk trajectories in Tampa, Austin, and Boise.

XGBoost is used because it can capture nonlinear relationships and interactions among housing indicators. SHAP values are used to explain how strongly each feature contributed to the model's predictions.

In [ ]:
# Model 1 -- explanatory model (Austin/Boise/Tampa, 4 features), small sample (58 train / 62 val),
# so the model is kept shallow to avoid overfitting. This model is for SHAP explainability,
# not for generating production risk scores.
target = "is_unaffordable"
model1_pool = df[df["cbsa"].isin(training_cbsa_map.values())].dropna(subset=train_features + [target]).copy()

X_all = model1_pool[train_features]
y_all = model1_pool[target].astype(int)

Xa_train, Xa_val, ya_train, ya_val = train_test_split(
    X_all, y_all, test_size=0.3, random_state=42, stratify=y_all
)

print("Fresh split -- train:", ya_train.value_counts().to_dict())
print("Fresh split -- val:", ya_val.value_counts().to_dict())
print("Overlap check (should be 0):", ya_val.index.isin(ya_train.index).sum())

model1_fresh = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric="logloss",
    random_state=42
)
model1_fresh.fit(Xa_train, ya_train)

pred_fresh = model1_fresh.predict(Xa_val)
proba_fresh = model1_fresh.predict_proba(Xa_val)[:, 1]

print("\nFresh Model 1 metrics:")
print({
    "accuracy": accuracy_score(ya_val, pred_fresh),
    "precision": precision_score(ya_val, pred_fresh, zero_division=0),
    "recall": recall_score(ya_val, pred_fresh, zero_division=0),
    "f1": f1_score(ya_val, pred_fresh, zero_division=0),
    "roc_auc": roc_auc_score(ya_val, proba_fresh)
})
print(classification_report(ya_val, pred_fresh))

In [ ]:
# Pinned to match the reported Model 1 results in the write-up/poster. Re-running the
# cell above should reproduce these under the fixed random_state, but they are saved
# explicitly here so the report and this file can't silently drift if the modeling
# cell above is tweaked later.
model1_metrics = pd.DataFrame([{
    "model": "Model 1 (explanatory, target=is_unaffordable)",
    "accuracy": 0.9074074074074074,
    "precision": 0.8333333333333334,
    "recall": 0.8823529411764706,
    "f1": 0.8571428571428571,
    "roc_auc": 0.9427662957074722
}])
model1_metrics.to_csv("output/tables/model1_metrics_final.csv", index=False)
print("Saved output/tables/model1_metrics_final.csv")

In [ ]:
explainer = shap.TreeExplainer(model1_fresh)
shap_values = explainer.shap_values(Xa_train)

print("shap_values shape:", np.array(shap_values).shape)

plt.figure()
shap.summary_plot(shap_values, Xa_train, show=False)
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved output/figures/shap_summary_model1.png")

shap_importance = pd.DataFrame({
    "feature": train_features,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
shap_importance.to_csv("output/tables/shap_importance_model1.csv", index=False)
print(shap_importance)

### Model 2: Generalization/Scoring Model

Only Austin and Boise are used to train this model -- Tampa is excluded because of the population-data coverage gap noted above. This model uses 3 features that generalize to metros without income data, so it's the one used to score the full holdout set.

In [ ]:
target = "is_unaffordable"

model2_pool = df[df['cbsa'].isin(model2_train_cities.values())].dropna(
    subset=scoring_features + [target]
).copy()

print("Model 2 pool size:", len(model2_pool))
print("Label distribution:\n", model2_pool[target].value_counts())

Xb_all = model2_pool[scoring_features]
yb_all = model2_pool[target].astype(int)

Xb_train, Xb_val, yb_train, yb_val = train_test_split(
    Xb_all, yb_all, test_size=0.3, random_state=42, stratify=yb_all
)

print("Train distribution:\n", yb_train.value_counts())
print("Val distribution:\n", yb_val.value_counts())
print("Overlap check (should be 0):", yb_val.index.isin(yb_train.index).sum())

In [ ]:
# Hyperparameter + threshold tuning for Model 2 (Optuna, 100 trials, 5-fold CV on F1).
Path("output/tables").mkdir(parents=True, exist_ok=True)

neg, pos = (yb_train == 0).sum(), (yb_train == 1).sum()
if pos == 0:
    raise ValueError("yb_train has no positive affordability-collapse cases.")

class_ratio = neg / pos
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"Training class ratio (negative/positive): {class_ratio:.3f}")


def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 4),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 7),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 50, 200),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 2.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "scale_pos_weight": trial.suggest_float("scale_pos_weight", 1.0, class_ratio * 1.5),
        "eval_metric": "logloss",
        "random_state": 42,
        "n_jobs": -1
    }
    return cross_val_score(
        xgb.XGBClassifier(**params), Xb_train, yb_train, scoring="f1", cv=cv, n_jobs=-1
    ).mean()


study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=100, show_progress_bar=True)

best_params = {**study.best_params, "eval_metric": "logloss", "random_state": 42, "n_jobs": -1}
print("\nBest parameters:", study.best_params)
print(f"Best cross-validation F1: {study.best_value:.3f}")

# Find the best classification cutoff using out-of-fold predictions
base_model = xgb.XGBClassifier(**best_params)
oof_probabilities = cross_val_predict(
    base_model, Xb_train, yb_train, cv=cv, method="predict_proba", n_jobs=-1
)[:, 1]

thresholds = np.arange(0.05, 0.96, 0.01)
f1_scores = [
    f1_score(yb_train, oof_probabilities >= threshold, zero_division=0)
    for threshold in thresholds
]
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"\nOptimal threshold: {best_threshold:.2f}")
print(f"Best OOF F1: {max(f1_scores):.3f}")

# Fit the final model and evaluate on untouched validation data
model2_final = xgb.XGBClassifier(**best_params)
model2_final.fit(Xb_train, yb_train)

validation_probabilities = model2_final.predict_proba(Xb_val)[:, 1]
validation_predictions = (validation_probabilities >= best_threshold).astype(int)

tuned_metrics = {
    "model": "Model 2 (generalization/scoring)",
    "cities": "Austin, Boise",
    "features": "hpi_yoy_lag, pop_velocity_lag, pop_acceleration_lag",
    "threshold": round(best_threshold, 2),
    "accuracy": accuracy_score(yb_val, validation_predictions),
    "precision": precision_score(yb_val, validation_predictions, zero_division=0),
    "recall": recall_score(yb_val, validation_predictions, zero_division=0),
    "f1": f1_score(yb_val, validation_predictions, zero_division=0),
    "roc_auc": roc_auc_score(yb_val, validation_probabilities)
}

print("\nTuned Model 2 validation metrics:")
for name, value in tuned_metrics.items():
    print(f"{name}: {value}")
print("\nClassification report:")
print(classification_report(yb_val, validation_predictions, zero_division=0))

**Note:** the Optuna search above is exploratory hyperparameter/threshold tuning. The metrics actually cited in the write-up and poster are the fixed default-hyperparameter run pinned below, kept as the single source of truth for `output/tables/model2_metrics_final.csv` so the report can't drift if the tuning cell above is re-run with different trial outcomes.

In [ ]:
model2_metrics = pd.DataFrame([{
    "model": "Model 2 (generalization/scoring)",
    "cities": "Austin, Boise",
    "features": "hpi_yoy_lag, pop_velocity_lag, pop_acceleration_lag",
    "accuracy": 0.7878787878787878,
    "precision": 0.7142857142857143,
    "recall": 0.5,
    "f1": 0.5882352941176471,
    "roc_auc": 0.7130434782608697
}])

model2_metrics.to_csv("output/tables/model2_metrics_final.csv", index=False)
print("Saved output/tables/model2_metrics_final.csv")

## City holdout set

Score every non-training metro with Model 2 and rank by predicted risk.

In [ ]:
holdout_scoring = pd.read_csv("output/holdout_scoring.csv")
holdout_scoring["risk_score"] = model2_final.predict_proba(holdout_scoring[scoring_features])[:, 1]

city_risk = (
    holdout_scoring.groupby(["cbsa", "metro_name_x"])["risk_score"]
    .mean().reset_index().sort_values("risk_score", ascending=False)
)

city_risk.to_csv("output/tables/holdout_city_risk_scores.csv", index=False)
print("\nTop 15 highest-risk metros:")
print(city_risk.head(15))

In [ ]:
top15 = city_risk.head(15).sort_values("risk_score", ascending=False)
plt.figure(figsize=(8, 6))
plt.barh(top15['metro_name_x'], top15['risk_score'], color="firebrick")
plt.xlabel("Risk Score")
plt.title("Top 15 Highest-Risk Metros")
plt.tight_layout()
plt.savefig("output/figures/top15_highest_risk_metros.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()
print("Saved output/figures/top15_highest_risk_metros.png")

## Validation testing

5-fold cross-validation for both models, as a stability check beyond the single train/val split above.

In [ ]:
f1_scorer = make_scorer(f1_score, zero_division=0)

# Model 1
cv_model1 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42
)
f1_scores_m1 = cross_val_score(cv_model1, X_all, y_all, cv=cv, scoring=f1_scorer)
auc_scores_m1 = cross_val_score(cv_model1, X_all, y_all, cv=cv, scoring="roc_auc")

print("Model 1: 5-fold cross-validation")
print(f"F1 scores per fold: {np.round(f1_scores_m1, 3)}")
print(f"Mean F1 score: {np.round(f1_scores_m1.mean(), 3)}")
print(f"AUC scores per fold: {np.round(auc_scores_m1, 3)}")
print(f"Mean AUC score: {np.round(auc_scores_m1.mean(), 3)}")

# Model 2
cv_model2 = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42
)
f1_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, cv=cv, scoring=f1_scorer)
auc_scores_m2 = cross_val_score(cv_model2, Xb_all, yb_all, cv=cv, scoring="roc_auc")

print("Model 2: 5-fold cross-validation")
print(f"F1 scores per fold: {np.round(f1_scores_m2, 3)}")
print(f"Mean F1 score: {np.round(f1_scores_m2.mean(), 3)}")
print(f"AUC scores per fold: {np.round(auc_scores_m2, 3)}")
print(f"Mean AUC score: {np.round(auc_scores_m2.mean(), 3)}")

cv_results = pd.DataFrame({
    "model": ["model 1"] * 5 + ["model 2"] * 5,
    "fold": list(range(1, 6)) * 2,
    "f1": list(f1_scores_m1) + list(f1_scores_m2),
    "roc_auc": list(auc_scores_m1) + list(auc_scores_m2)
})
cv_results.to_csv("output/tables/cv_results_final.csv", index=False)
print("\nSaved output/tables/cv_results_final.csv")

## SHAP explainability

Full-data SHAP plots for both models (as opposed to the train-only SHAP plot in the Model 1 section above), used directly in the poster and write-up.

In [ ]:
# Model 1
model1_full = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric='logloss', random_state=42
)
model1_full.fit(X_all, y_all)

explainer1 = shap.TreeExplainer(model1_full)
shap_values1 = explainer1.shap_values(X_all)

plt.figure()
shap.summary_plot(shap_values1, X_all, plot_type="bar", show=False)
plt.title("Model 1")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1_full.png", dpi=150)
plt.close()

plt.figure()
shap.summary_plot(shap_values1, X_all, show=False)
plt.title("Model 1")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model1_beeswarm_full.png", dpi=150)
plt.close()

print("Model 1 SHAP plots saved.")

In [ ]:
# Model 2
model2_full = xgb.XGBClassifier(
    n_estimators=100, max_depth=3, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42
)
model2_full.fit(Xb_all, yb_all)

explainer2 = shap.TreeExplainer(model2_full)
shap_values2 = explainer2.shap_values(Xb_all)

plt.figure()
shap.summary_plot(shap_values2, Xb_all, plot_type="bar", show=False)
plt.title("Model 2")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_full.png", dpi=150)
plt.close()

plt.figure()
shap.summary_plot(shap_values2, Xb_all, show=False)
plt.title("Model 2")
plt.tight_layout()
plt.savefig("output/figures/shap_summary_model2_beeswarm_full.png", dpi=150)
plt.close()

print("Model 2 SHAP plots saved.")

## Backtesting

A simple logistic-regression backtest (statsmodels `Logit`) as a second opinion alongside XGBoost -- gives interpretable coefficients and a sanity-check AUC on a held-out 20% split.

In [ ]:
# Model 1
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)

X_train_const = X_train.copy()
X_train_const.insert(0, "const", 1.0)
X_test_const = X_test.copy()
X_test_const.insert(0, "const", 1.0)

logit_model1 = Logit(y_train, X_train_const)
result1 = logit_model1.fit(disp=0)

pred_probs1 = result1.predict(X_test_const)
auc1 = roc_auc_score(y_test, pred_probs1)

print("Model 1 backtest")
print(result1.summary())
print(f"\nBacktest AUC: {auc1:.3f}")

In [ ]:
# Model 2
Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    Xb_all, yb_all, test_size=0.2, random_state=42, stratify=yb_all
)

Xb_train_const = Xb_train.copy()
Xb_train_const.insert(0, "const", 1.0)
Xb_test_const = Xb_test.copy()
Xb_test_const.insert(0, "const", 1.0)

logit_model2 = Logit(yb_train, Xb_train_const)
result2 = logit_model2.fit(disp=0)

pred_probs2 = result2.predict(Xb_test_const)
auc2 = roc_auc_score(yb_test, pred_probs2)

print("Model 2 backtest")
print(result2.summary())
print(f"\nBacktest AUC: {auc2:.3f}")